In [1]:
from pathlib import Path
import importlib
import sys

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import utils.common as common_module
import utils.data as data_module
import utils.film_routing as film_module
import utils.losses as losses_module

common_module = importlib.reload(common_module)
data_module = importlib.reload(data_module)
film_module = importlib.reload(film_module)
losses_module = importlib.reload(losses_module)

BCEDiceLoss = losses_module.BCEDiceLoss
DEFAULT_CUBE_SIZES = data_module.DEFAULT_CUBE_SIZES
BereaPatchDataset = data_module.BereaPatchDataset
FiLMRoutedUNet3D = film_module.FiLMRoutedUNet3D
auxiliary_physics_loss = losses_module.auxiliary_physics_loss
dice_score_from_logits = losses_module.dice_score_from_logits
embedding_consistency_loss = losses_module.embedding_consistency_loss
from utils.training import EarlyStopping, MetricTracker

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "device:", device)


ROOT: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct device: cuda


In [2]:
TRAIN_MODE = "quick"  # quick | full
CUBE_SIZES = list(DEFAULT_CUBE_SIZES)
BATCH_SIZE = 1
EPOCHS = 10
LR = 1e-4
AUX_WEIGHT = 0.05
CONSISTENCY_WEIGHT = 0.05
PATIENCE = 3
MIN_DELTA = 1e-4
BASE_CHANNELS = 16
CTX_DIM = 64
SAMPLES_PER_GROUP = 8 if TRAIN_MODE == "quick" else None
MAX_TRAIN_BATCHES = 64 if TRAIN_MODE == "quick" else None
MAX_VAL_BATCHES = 16 if TRAIN_MODE == "quick" else None
SIZE_SAMPLING_WEIGHTS = {64: 0.50, 128: 0.35, 192: 0.15}
NUM_WORKERS = 0  # Windows/Jupyter: multiprocessing DataLoader often fails with spawn/pickle errors
PIN_MEMORY = device == "cuda"
USE_AMP = device == "cuda"
SAVE_CHECKPOINT = True
CHECKPOINT_PATH = MODEL_DIR / "film_routed_unet3d_best.pth"


In [3]:
train_ds = BereaPatchDataset(
    ROOT,
    split="train",
    cube_size=CUBE_SIZES,
    use_raw_gray=False,
    balance=True,
    samples_per_group=SAMPLES_PER_GROUP,
    size_sampling_weights=SIZE_SAMPLING_WEIGHTS,
)
val_ds = BereaPatchDataset(
    ROOT,
    split="val",
    cube_size=CUBE_SIZES,
    use_raw_gray=False,
    noise_types=["none"],
    balance=False,
    samples_per_group=SAMPLES_PER_GROUP,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
if NUM_WORKERS != 0:
    print("warning: num_workers>0 is unreliable in Windows notebooks; prefer a .py training script")
print("mode:", TRAIN_MODE, "cube sizes:", CUBE_SIZES)
print("train:", len(train_ds), "val:", len(val_ds), "max batches:", MAX_TRAIN_BATCHES, MAX_VAL_BATCHES)
if hasattr(train_ds, "df"):
    display(train_ds.df.groupby(["rock", "cube_size", "split"]).size().rename("samples").reset_index())


mode: quick cube sizes: [64, 128, 192]
train: 48 val: 48 max batches: 64 16


,rock,cube_size,split,samples
0,BanderaBrown,64,train,8
1,BanderaBrown,128,train,8
2,BanderaBrown,192,train,8
3,Berea,64,train,8
4,Berea,128,train,8
5,Berea,192,train,8


In [4]:
model = FiLMRoutedUNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    ctx_dim=CTX_DIM,
).to(device)
criterion = BCEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
print("parameters:", sum(p.numel() for p in model.parameters()))


parameters: 1484461


In [5]:
early_stopping = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA, mode="min")
history = []
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_stats = MetricTracker()
    train_bar = tqdm(train_loader, desc=f"train {epoch}/{EPOCHS}", leave=False)
    for batch_idx, batch in enumerate(train_bar):
        if MAX_TRAIN_BATCHES is not None and batch_idx >= MAX_TRAIN_BATCHES:
            break
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out = model(x, return_dict=True)
            logits = out["logits"]
            seg_loss, bce_loss, dice_loss = criterion(logits, y)
            aux_loss, aux_parts = auxiliary_physics_loss(
                out,
                y,
                porosity_target=batch["porosity"].to(device),
                percolation_target=batch["percolates"].to(device),
                porosity_weight=AUX_WEIGHT,
                percolation_weight=AUX_WEIGHT,
            )
            loss = seg_loss + aux_loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = x.size(0)
        with torch.no_grad():
            dice = dice_score_from_logits(logits, y)
        train_stats.update("loss", float(loss.detach().cpu()), batch_size)
        train_stats.update("seg_loss", float(seg_loss.detach().cpu()), batch_size)
        train_stats.update("aux_loss", float(aux_loss.detach().cpu()), batch_size)
        train_stats.update("bce", float(bce_loss.detach().cpu()), batch_size)
        train_stats.update("dice_loss", float(dice_loss.detach().cpu()), batch_size)
        train_stats.update("dice", float(dice.detach().cpu()), batch_size)
        train_bar.set_postfix(train_stats.postfix("loss", "bce", "dice_loss", "dice"))

    model.eval()
    val_stats = MetricTracker()
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"val {epoch}/{EPOCHS}", leave=False)
        for batch_idx, batch in enumerate(val_bar):
            if MAX_VAL_BATCHES is not None and batch_idx >= MAX_VAL_BATCHES:
                break
            x = batch["x"].to(device)
            y = batch["y"].to(device)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(x, return_dict=True)
                logits = out["logits"]
                seg_loss, bce_loss, dice_loss = criterion(logits, y)
                aux_loss, aux_parts = auxiliary_physics_loss(
                    out,
                    y,
                    porosity_target=batch["porosity"].to(device),
                    percolation_target=batch["percolates"].to(device),
                    porosity_weight=AUX_WEIGHT,
                    percolation_weight=AUX_WEIGHT,
                )
                loss = seg_loss + aux_loss
            dice = dice_score_from_logits(logits, y)

            batch_size = x.size(0)
            val_stats.update("loss", float(loss.detach().cpu()), batch_size)
            val_stats.update("seg_loss", float(seg_loss.detach().cpu()), batch_size)
            val_stats.update("aux_loss", float(aux_loss.detach().cpu()), batch_size)
            val_stats.update("bce", float(bce_loss.detach().cpu()), batch_size)
            val_stats.update("dice_loss", float(dice_loss.detach().cpu()), batch_size)
            val_stats.update("dice", float(dice.detach().cpu()), batch_size)
            val_bar.set_postfix(val_stats.postfix("loss", "bce", "dice_loss", "dice"))

    train_metrics = train_stats.as_dict()
    val_metrics = val_stats.as_dict()
    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_dice": train_metrics["dice"],
        "val_loss": val_metrics["loss"],
        "val_dice": val_metrics["dice"],
    })

    print(
        f"epoch={epoch} "
        f"train_loss={train_metrics['loss']:.4f} train_dice={train_metrics['dice']:.4f} "
        f"val_loss={val_metrics['loss']:.4f} val_dice={val_metrics['dice']:.4f}"
    )
    print("alpha:", model.router.alpha().detach().cpu())

    improved = early_stopping.step(val_metrics["loss"], epoch=epoch)
    if improved and SAVE_CHECKPOINT:
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "val_loss": val_metrics["loss"],
            "val_dice": val_metrics["dice"],
            "history": history,
            "base_channels": BASE_CHANNELS,
            "ctx_dim": CTX_DIM,
        }, CHECKPOINT_PATH)
        print("сохранено:", CHECKPOINT_PATH)
    elif improved:
        print("architecture smoke: checkpoint skipped")
    else:
        print(f"ранняя остановка: {early_stopping.bad_epochs}/{PATIENCE} эпох без улучшения val_loss")

    if early_stopping.should_stop:
        print(f"обучение остановлено на эпохе {epoch}; лучший val_loss={early_stopping.best:.4f}")
        break


C:\Users\F\AppData\Local\Temp\ipykernel_24832\3772087896.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
train 1/10:   0%|          | 0/48 [00:00<?, ?it/s]C:\Users\F\AppData\Local\Temp\ipykernel_24832\3772087896.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
val 1/10:   0%|          | 0/48 [00:00<?, ?it/s]                                                                   C:\Users\F\AppData\Local\Temp\ipykernel_24832\3772087896.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


epoch=1 train_loss=0.5839 train_dice=0.7244 val_loss=0.4980 val_dice=0.8578
alpha: tensor([[0.9480, 0.0173, 0.0174, 0.0173],
        [0.0174, 0.9479, 0.0174, 0.0174],
        [0.0174, 0.0174, 0.9477, 0.0174],
        [0.0175, 0.0175, 0.0175, 0.9476]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=2 train_loss=0.4534 train_dice=0.8921 val_loss=0.4172 val_dice=0.9061
alpha: tensor([[0.9479, 0.0174, 0.0174, 0.0173],
        [0.0174, 0.9479, 0.0173, 0.0174],
        [0.0174, 0.0174, 0.9477, 0.0174],
        [0.0177, 0.0174, 0.0177, 0.9472]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=3 train_loss=0.3706 train_dice=0.9236 val_loss=0.3364 val_dice=0.9334
alpha: tensor([[0.9480, 0.0173, 0.0174, 0.0173],
        [0.0174, 0.9478, 0.0174, 0.0174],
        [0.0174, 0.0174, 0.9477, 0.0174],
        [0.0180, 0.0174, 0.0180, 0.9466]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=4 train_loss=0.2783 train_dice=0.9332 val_loss=0.2496 val_dice=0.9475
alpha: tensor([[0.9479, 0.0174, 0.0174, 0.0173],
        [0.0174, 0.9479, 0.0173, 0.0174],
        [0.0174, 0.0174, 0.9478, 0.0174],
        [0.0183, 0.0174, 0.0183, 0.9459]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


...

epoch=6 train_loss=0.1374 train_dice=0.9411 val_loss=0.1329 val_dice=0.9553
alpha: tensor([[0.9479, 0.0174, 0.0174, 0.0174],
        [0.0174, 0.9477, 0.0174, 0.0174],
        [0.0175, 0.0174, 0.9477, 0.0174],
        [0.0189, 0.0175, 0.0190, 0.9446]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=7 train_loss=0.1057 train_dice=0.9451 val_loss=0.1122 val_dice=0.9487
alpha: tensor([[0.9480, 0.0173, 0.0174, 0.0173],
        [0.0175, 0.9476, 0.0174, 0.0174],
        [0.0175, 0.0175, 0.9476, 0.0175],
        [0.0192, 0.0175, 0.0192, 0.9442]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=8 train_loss=0.0982 train_dice=0.9415 val_loss=0.0985 val_dice=0.9469
alpha: tensor([[0.9480, 0.0173, 0.0174, 0.0173],
        [0.0175, 0.9476, 0.0175, 0.0174],
        [0.0175, 0.0175, 0.9476, 0.0175],
        [0.0193, 0.0175, 0.0194, 0.9438]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=9 train_loss=0.0845 train_dice=0.9452 val_loss=0.0796 val_dice=0.9604
alpha: tensor([[0.9478, 0.0174, 0.0174, 0.0174],
        [0.0175, 0.9476, 0.0175, 0.0174],
        [0.0175, 0.0175, 0.9475, 0.0175],
        [0.0195, 0.0175, 0.0195, 0.9434]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=10 train_loss=0.0761 train_dice=0.9491 val_loss=0.0711 val_dice=0.9641
alpha: tensor([[0.9478, 0.0174, 0.0174, 0.0174],
        [0.0175, 0.9476, 0.0175, 0.0174],
        [0.0175, 0.0175, 0.9475, 0.0175],
        [0.0196, 0.0176, 0.0197, 0.9431]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth
